In [1]:
# Install required packages for the project
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install scikit-learn pandas numpy matplotlib seaborn umap-learn
%pip install captum plotly astropy tqdm
%pip install --upgrade --force-reinstall numpy pandas

# Optional: Install ztfquery for real ZTF data (requires IRSA account)
# pip install ztfquery

print("✅ All packages installed successfully!")

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.4.0-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cache

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
captum 0.8.0 requires numpy<2.0, but you have numpy 2.4.0 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.0 which is incompatible.
numba 0.59.1 requires numpy<1.27,>=1.22, but you have numpy 2.4.0 which is incompatible.
pywavelets 1.5.0 requires numpy<2.0,>=1.22.4, but you have numpy 2.4.0 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.4.0 which is incompatible.
streamlit 1.32.0 requires numpy<2,>=1.19.3, but you have numpy 2.4.0 which is incompatible.


In [2]:
"""
Complete ZTF Light Curve Dataset Downloader (FIXED)
Downloads multi-band light curves for rare astronomical transients
Optimized for contrastive learning research
"""

import requests
import pandas as pd
import numpy as np
import time
import json
from typing import List, Dict, Tuple
from datetime import datetime
import os

class ZTFDatasetDownloader:
    """
    Download and preprocess ZTF light curve datasets for rare transient classification
    """
    
    def __init__(self, output_dir: str = 'ztf_dataset'):
        self.base_url = "https://api.alerce.online/ztf/v1"
        self.output_dir = output_dir
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        print(f"Output directory: {output_dir}")
        
    def search_objects(self, 
                      classifier: str = None,
                      class_name: str = None,
                      page: int = 1,
                      page_size: int = 100) -> Dict:
        """
        Search for ZTF objects matching criteria
        
        Args:
            classifier: Classifier name ('lc_classifier', 'stamp_classifier')
            class_name: Transient class
            page: Page number for pagination
            page_size: Number of results per page (max 100)
        
        Returns:
            Dictionary with search results
        """
        url = f"{self.base_url}/objects"
        
        params = {
            'page': page,
            'page_size': page_size,
            'order_by': 'ndet',
            'order_mode': 'DESC'
        }
        
        # Add classifier and class filter if provided
        if classifier and class_name:
            params['classifier'] = classifier
            params['class'] = class_name
            
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            return {'items': [], 'total': 0}
    
    def get_classifier_classes(self) -> List[str]:
        """
        Get available classifier classes from the API
        
        Returns:
            List of available class names
        """
        url = f"{self.base_url}/classifiers"
        
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            classifiers = response.json()
            
            # Extract class names from all classifiers
            all_classes = set()
            for classifier in classifiers:
                if 'classes' in classifier:
                    all_classes.update(classifier['classes'])
            
            return sorted(list(all_classes))
        except Exception as e:
            print(f"Error fetching classifiers: {e}")
            return []
    
    def get_object_ids(self, 
                      class_name: str = None,
                      n_objects: int = 100,
                      classifier: str = 'lc_classifier') -> List[str]:
        """
        Get list of object IDs for a specific class
        
        Args:
            class_name: Transient class name (None for all objects)
            n_objects: Number of objects to fetch
            classifier: Classifier to use
        
        Returns:
            List of ZTF object IDs
        """
        object_ids = []
        page = 1
        
        class_str = class_name if class_name else "All"
        print(f"\nSearching for {class_str} objects...")
        
        while len(object_ids) < n_objects:
            result = self.search_objects(
                classifier=classifier if class_name else None,
                class_name=class_name,
                page=page,
                page_size=100
            )
            
            items = result.get('items', [])
            if not items:
                print(f"No more objects found. Got {len(object_ids)} total.")
                break
            
            for item in items:
                oid = item.get('oid')
                ndet = item.get('ndet', 0)
                
                # Filter by minimum detections
                if oid and ndet >= 20:
                    object_ids.append(oid)
                    
                if len(object_ids) >= n_objects:
                    break
            
            print(f"  Collected {len(object_ids)}/{n_objects} objects (page {page})", end='\r')
            page += 1
            time.sleep(0.5)  # Rate limiting
            
            # Safety check - don't paginate forever
            if page > 50:
                break
        
        print(f"\nFound {len(object_ids)} {class_str} objects with sufficient detections")
        return object_ids[:n_objects]
    
    def get_light_curve(self, oid: str) -> Tuple[pd.DataFrame, Dict]:
        """
        Fetch light curve detections and metadata for an object
        
        Args:
            oid: ZTF object identifier
        
        Returns:
            Tuple of (light_curve_df, metadata_dict)
        """
        # Get detections
        det_url = f"{self.base_url}/objects/{oid}/detections"
        
        # Get metadata
        meta_url = f"{self.base_url}/objects/{oid}"
        
        try:
            # Fetch detections
            det_response = requests.get(det_url, timeout=20)
            det_response.raise_for_status()
            detections = det_response.json()
            
            # Fetch metadata
            meta_response = requests.get(meta_url, timeout=20)
            meta_response.raise_for_status()
            metadata = meta_response.json()
            
            # Process detections into DataFrame
            if detections:
                lc_data = []
                for det in detections:
                    lc_data.append({
                        'oid': oid,
                        'mjd': det.get('mjd'),
                        'fid': det.get('fid'),  # 1=g-band, 2=r-band
                        'mag': det.get('mag'),
                        'e_mag': det.get('e_mag'),
                        'magpsf': det.get('magpsf'),
                        'sigmapsf': det.get('sigmapsf'),
                        'ra': det.get('ra'),
                        'dec': det.get('dec'),
                        'isdiffpos': det.get('isdiffpos')
                    })
                
                df = pd.DataFrame(lc_data)
                
                # Filter only g and r bands (fid 1 and 2)
                df = df[df['fid'].isin([1, 2])]
                
                # Sort by time
                df = df.sort_values('mjd').reset_index(drop=True)
                
                return df, metadata
            else:
                return pd.DataFrame(), metadata
                
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {oid}: {e}")
            return pd.DataFrame(), {}
    
    def download_class_dataset(self,
                               class_name: str = None,
                               n_objects: int = 100,
                               min_detections: int = 20,
                               classifier: str = 'lc_classifier',
                               save_individual: bool = False) -> pd.DataFrame:
        """
        Download complete dataset for a transient class
        
        Args:
            class_name: Transient class name (None for diverse sample)
            n_objects: Number of objects to download
            min_detections: Minimum detections required
            classifier: Classifier to use
            save_individual: Save individual light curves to separate files
        
        Returns:
            Combined DataFrame with all light curves
        """
        class_str = class_name if class_name else "Mixed"
        print(f"\n{'='*60}")
        print(f"Downloading {class_str} dataset")
        print(f"{'='*60}")
        
        # Get object IDs
        object_ids = self.get_object_ids(class_name, n_objects, classifier)
        
        if not object_ids:
            print(f"No objects found for {class_str}")
            return pd.DataFrame()
        
        # Download light curves
        all_lcs = []
        metadata_list = []
        successful = 0
        
        print(f"\nDownloading light curves...")
        
        for i, oid in enumerate(object_ids):
            print(f"  Progress: {i+1}/{len(object_ids)} - {oid} - Success: {successful}", end='\r')
            
            lc_df, metadata = self.get_light_curve(oid)
            
            if not lc_df.empty and len(lc_df) >= min_detections:
                # Extract class from metadata
                if class_name:
                    lc_df['class'] = class_name
                else:
                    # Try to get class from metadata
                    classifiers = metadata.get('probabilities', {})
                    if classifiers:
                        lc_classifier = classifiers.get('lc_classifier', {})
                        if lc_classifier:
                            predicted_class = max(lc_classifier.items(), key=lambda x: x[1])[0]
                            lc_df['class'] = predicted_class
                        else:
                            lc_df['class'] = 'Unknown'
                    else:
                        lc_df['class'] = 'Unknown'
                
                all_lcs.append(lc_df)
                
                metadata['assigned_class'] = lc_df['class'].iloc[0]
                metadata_list.append(metadata)
                
                # Save individual light curve if requested
                if save_individual:
                    safe_class = lc_df['class'].iloc[0].replace('/', '_')
                    lc_file = os.path.join(self.output_dir, f"{safe_class}_{oid}.csv")
                    lc_df.to_csv(lc_file, index=False)
                
                successful += 1
            
            # Rate limiting
            time.sleep(0.5)
        
        print(f"\n\nSuccessfully downloaded {successful}/{len(object_ids)} light curves")
        
        # Combine all light curves
        if all_lcs:
            combined_df = pd.concat(all_lcs, ignore_index=True)
            
            # Save combined dataset
            safe_class_name = (class_name.replace('/', '_') if class_name else 'mixed')
            output_file = os.path.join(self.output_dir, f'{safe_class_name}_lightcurves.csv')
            combined_df.to_csv(output_file, index=False)
            print(f"Saved to: {output_file}")
            
            # Save metadata
            meta_file = os.path.join(self.output_dir, f'{safe_class_name}_metadata.json')
            with open(meta_file, 'w') as f:
                json.dump(metadata_list, f, indent=2)
            
            return combined_df
        else:
            return pd.DataFrame()
    
    def download_diverse_dataset(self,
                                total_objects: int = 300,
                                min_detections: int = 20) -> pd.DataFrame:
        """
        Download a diverse dataset without specifying classes
        (Lets the API return various transient types)
        
        Args:
            total_objects: Total number of objects to download
            min_detections: Minimum detections per object
        
        Returns:
            Combined DataFrame with diverse transient types
        """
        print(f"\n{'='*60}")
        print(f"DOWNLOADING DIVERSE TRANSIENT DATASET")
        print(f"{'='*60}")
        print(f"Target: {total_objects} objects with {min_detections}+ detections")
        
        dataset = self.download_class_dataset(
            class_name=None,  # Get diverse sample
            n_objects=total_objects,
            min_detections=min_detections,
            classifier=None  # Don't filter by classifier
        )
        
        if not dataset.empty:
            # Print class distribution
            print(f"\n{'='*60}")
            print("CLASS DISTRIBUTION IN DOWNLOADED DATA")
            print(f"{'='*60}")
            class_counts = dataset.groupby('class')['oid'].nunique().sort_values(ascending=False)
            for class_name, count in class_counts.items():
                print(f"  {class_name}: {count} objects")
            
            # Save summary
            output_file = os.path.join(self.output_dir, 'diverse_dataset.csv')
            dataset.to_csv(output_file, index=False)
            print(f"\nDataset saved to: {output_file}")
            
        return dataset
    
    def explore_sample_objects(self, n_samples: int = 10) -> pd.DataFrame:
        """
        Download a small sample to explore data structure
        
        Args:
            n_samples: Number of sample objects
        
        Returns:
            Sample DataFrame
        """
        print(f"Downloading {n_samples} sample objects...")
        
        # Get random objects
        result = self.search_objects(page=1, page_size=n_samples)
        items = result.get('items', [])
        
        if not items:
            print("No objects found")
            return pd.DataFrame()
        
        samples = []
        
        for i, item in enumerate(items[:n_samples]):
            oid = item.get('oid')
            print(f"  Fetching {i+1}/{n_samples}: {oid}", end='\r')
            
            lc_df, metadata = self.get_light_curve(oid)
            
            if not lc_df.empty:
                # Get class from metadata
                classifiers = metadata.get('probabilities', {})
                if classifiers:
                    lc_classifier = classifiers.get('lc_classifier', {})
                    if lc_classifier:
                        predicted_class = max(lc_classifier.items(), key=lambda x: x[1])[0]
                        lc_df['class'] = predicted_class
                
                samples.append(lc_df)
            
            time.sleep(0.5)
        
        if samples:
            sample_df = pd.concat(samples, ignore_index=True)
            print(f"\n\nSample data structure:")
            print(sample_df.head())
            print(f"\nClasses found: {sample_df['class'].unique()}")
            return sample_df
        else:
            return pd.DataFrame()


# =============================================================================
# SIMPLIFIED EXECUTION FUNCTIONS
# =============================================================================

def quick_diverse_download(n_objects: int = 100):
    """
    Quick start: Download diverse transients without specifying classes
    This is the RECOMMENDED approach for your research
    """
    print("="*60)
    print("QUICK START - DIVERSE TRANSIENT DOWNLOAD")
    print("="*60)
    print("\nThis will download objects of various types automatically")
    print("The API will return a mix of common and rare transients\n")
    
    downloader = ZTFDatasetDownloader(output_dir='ztf_diverse_data')
    
    dataset = downloader.download_diverse_dataset(
        total_objects=n_objects,
        min_detections=20
    )
    
    if not dataset.empty:
        print("\n" + "="*60)
        print("DOWNLOAD COMPLETE - SUMMARY")
        print("="*60)
        print(f"Total objects: {dataset['oid'].nunique()}")
        print(f"Total detections: {len(dataset)}")
        
        print("\nFilter distribution:")
        filter_map = {1: 'g-band', 2: 'r-band'}
        print(dataset['fid'].map(filter_map).value_counts())
        
        print("\nDetections per object:")
        dets_per_obj = dataset.groupby('oid').size()
        print(f"  Min: {dets_per_obj.min()}")
        print(f"  Max: {dets_per_obj.max()}")
        print(f"  Mean: {dets_per_obj.mean():.1f}")
        
        print("\n✓ Dataset ready for contrastive learning!")
        print(f"  File: ztf_diverse_data/diverse_dataset.csv")
        
        return dataset
    else:
        print("\nDownload failed. Check your internet connection.")
        return None


def explore_api():
    """
    Explore the API and data structure with a small sample
    """
    print("="*60)
    print("EXPLORING ZTF API AND DATA STRUCTURE")
    print("="*60)
    
    downloader = ZTFDatasetDownloader(output_dir='ztf_exploration')
    
    # Download small sample
    sample = downloader.explore_sample_objects(n_samples=5)
    
    if not sample.empty:
        print("\n✓ Sample downloaded successfully!")
        print(f"  Check: ztf_exploration/ directory")
        
        # Show structure
        print("\nDataFrame columns:")
        print(sample.columns.tolist())
        
        print("\nSample light curve (first object):")
        first_oid = sample['oid'].iloc[0]
        print(sample[sample['oid'] == first_oid][['mjd', 'fid', 'mag', 'e_mag']].head(10))
        
        return sample
    else:
        print("\nExploration failed.")
        return None


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    
    print("\nStarting download in 3 seconds...")
    print("(This will download ~100 diverse transients)\n")
    time.sleep(3)
    
    dataset = quick_diverse_download(n_objects=100)
    



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\Lib\site-pack

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




Starting download in 3 seconds...
(This will download ~100 diverse transients)

QUICK START - DIVERSE TRANSIENT DOWNLOAD

This will download objects of various types automatically
The API will return a mix of common and rare transients

Output directory: ztf_diverse_data

DOWNLOADING DIVERSE TRANSIENT DATASET
Target: 100 objects with 20+ detections


Searching for All objects...
  Collected 100/100 objects (page 1)
Found 100 All objects with sufficient detections

Error fetching ZTF18aavepno: HTTPSConnectionPool(host='api.alerce.online', port=443): Max retries exceeded with url: /ztf/v1/objects/ZTF18aavepno/detections (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000023F8094CB00>, 'Connection to api.alerce.online timed out. (connect timeout=20)'))
Error fetching ZTF17aacsclx: HTTPSConnectionPool(host='api.alerce.online', port=443): Max retries exceeded with url: /ztf/v1/objects/ZTF17aacsclx (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConne

### Select 20000 rows to inital model training

In [3]:
import pandas as pd
from typing import List, Optional

def save_first_n_rows(
    input_csv: str,
    output_csv: str,
    drop_cols: Optional[List[str]] = None,
    n: int = 20000,
    **read_csv_kwargs
) -> str:
    """
    Load a CSV file, drop selected columns, and save the first n rows.

    Parameters
    ----------
    input_csv : str
        Path to input CSV file.
    output_csv : str
        Path to save the output CSV.
    drop_cols : list[str], optional
        Columns to drop.
    n : int
        Number of rows to keep.
    read_csv_kwargs : dict
        Extra arguments for pandas.read_csv().

    Returns
    -------
    str
        Path to the saved CSV file.
    """

    df = pd.read_csv(input_csv, **read_csv_kwargs)

    if drop_cols:
        cols_to_drop = [c for c in drop_cols if c in df.columns]
        df = df.drop(columns=cols_to_drop)

    df.head(n).to_csv(output_csv, index=False)
    return output_csv


In [4]:
save_first_n_rows(
    input_csv="diverse_dataset.csv",
    output_csv="light_curves.csv",
    drop_cols=["class"],
    n=20000
)

FileNotFoundError: [Errno 2] No such file or directory: 'diverse_dataset.csv'

### DATA SET OVERVIEW

# 📊 Dataset Description: Astronomical Light Curve Data

This dataset contains **time-series photometric observations of astronomical objects** collected from a sky survey. Each row represents **one observation at a specific time**, capturing how the brightness of an object changes over time in different filters.  
The dataset is used to support **unsupervised representation learning using SimCLR** for astronomical transient analysis.

---

## 🧾 Dataset Overview

- **File name:** `light_curves.csv`
- **Data type:** Tabular time-series data
- **Domain:** Astronomy / Time-domain astrophysics
- **Primary use:** Self-supervised contrastive learning (SimCLR)

---

## 📌 Feature Description

### 🕒 `mjd` — Modified Julian Date
- Represents the exact time at which the observation was recorded.
- Enables modeling of **temporal variations** in object brightness.
- Essential for learning **light curve evolution patterns**.


---

### 🎛️ `fid` — Filter ID
- Identifies the photometric filter used during observation (different wavelength bands).
- Objects exhibit different brightness levels across filters.
- Treated as a **categorical feature** and one-hot encoded.


---

### ✨ `magpsf` — PSF Magnitude
- Measures the brightness of the object using point-spread-function fitting.
- Core signal used to characterize light curves.
- Primary feature for detecting and learning transient behavior.


---

### 📉 `sigmapsf` — Magnitude Uncertainty
- Indicates the measurement error associated with `magpsf`.
- Helps the model account for **observational noise**.
- Encourages learning **robust and reliable embeddings**.


---

### 🌍 `ra` — Right Ascension
- Sky coordinate representing the object’s horizontal position.
- Provides spatial context within the celestial sphere.


---

### 🌍 `dec` — Declination
- Sky coordinate representing the object’s vertical position.
- Used together with `ra` to locate the object in the sky.


---

### 🔄 `isdiffpos` — Difference Image Detection Flag
- Indicates whether the object was detected as a positive signal in difference imaging.
- Values:
  - `1` → Positive detection  
  - `0` → Negative or non-significant detection
- Helps distinguish real transient events from background noise.


---

## 🧠 Relevance to the Research

This dataset is well-suited for **SimCLR-based contrastive learning** because it:

- Contains rich numerical and categorical features
- Captures temporal, spectral, spatial, and uncertainty information
- Does not require labeled data for representation learning
- Supports downstream tasks such as classification and clustering

---

## 🔬 Role in Methodology

In this research:
- The dataset is used to generate two augmented views of each observation.
- A SimCLR model learns meaningful embeddings in an unsupervised manner.
- The learned embeddings are later used for:
  - Transient classification
  - Clustering and visualization
  - Performance comparison with other learning approaches

---

## 📎 Summary

The light curve dataset provides a comprehensive foundation for astronomical time-series analysis by combining:

- ⏱️ Temporal information  
- ✨ Photometric brightness  
- 📉 Measurement uncertainty  
- 🌍 Spatial positioning  
- 🔄 Detection reliability  

These characteristics enable effective **self-supervised representation learning** for astronomical transient research.
